# GFTM B∥: do large-B∥ cases have bad growth rates, and does a 1/ky correction of the GFTM/GS2 B∥ ratio remove it? (Fusion_PhD-38e9.26)

Existing pyro objects only (GS2 `cube.nc` / `pyroscan.nc`, GFTM FILTER 0.5 leaves at WIDTH 0.4 and 0.8); no new runs.
Dominant mode = argmax γ over `mode` in each code. Per case, from the dominant mode's eigenfunction:

* r = peak|B∥| / peak|φ| (as `38e9.9`), and r₂ = (∫|B∥|²dθ / ∫|φ|²dθ)^½ as a robustness check; R = r_GFTM / r_GS2.
* e = log10(γ_GFTM / γ_GS2), and |Δγ|.

Populations: all GS2-unstable cases, and the GS2-KBM subset (`general_analysis.mode_classification.classify`, the user's
cutoffs in Fusion_PhD `docs/mode_filtering.md`). "Same mode" (for the ky fit): γ within ×1.5 and φ overlap > 0.85.

In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from scipy import stats
from scipy.integrate import trapezoid
from dotenv import load_dotenv
import pyrokinetics
from pyrokinetics import PyroScan
from pyrokinetics.pyroscan import PyroScanGKOutput
from general_analysis import mode_classification as mc

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
warnings.filterwarnings("ignore")
print("pyrokinetics", pyrokinetics.__version__)

analysis_name = "gftm_bpar_ratio"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
plot_dir = ROOT / "Plots" / analysis_name
plot_dir.mkdir(parents=True, exist_ok=True)

hypercubes = ["SPR-045", "M1"]               # LATIN_HYPERCUBE/<db>/kbm_8d
beta_scans = ["R3", "R4", "M1_A1_NE"]        # scan_beta (GS2) / scan_beta_f05 (GFTM)
databases = hypercubes + beta_scans
widths = {0.4: ("wo_f05_w0p4000", "w0p4"), 0.8: ("wo_f05_w0p8000", "w0p8")}   # (hypercube leaf, beta-scan leaf)
gs2_tol = 0.1          # beta scans: drop GS2 cells with growth_rate_tolerance >= this (as 38e9.9); hypercubes use the tail average
same_gamma = 1.5       # same mode: gamma ratio within x1.5 ...
same_overlap = 0.85    # ... and |<phi_GFTM, phi_GS2>| above this
top = 0.75             # "large B_par" = top quartile within a database
fs_title, fs_label, fs_tick, fs_legend = 16, 16, 14, 12

## Load: GS2 reference and GFTM dominant mode, one row per case

In [ ]:
def dominant(g, ef):
    "argmax gamma over the last (mode) axis; gamma 0 where GFTM has no unstable mode"
    g = np.nan_to_num(g, nan=-np.inf)
    i = g.argmax(-1)
    gd = np.take_along_axis(g, i[..., None], -1)[..., 0]
    efd = np.take_along_axis(ef, i[..., None, None, None], -1)[..., 0]
    return np.where(gd > 0, gd, 0.0), efd


def field_ratios(theta, ef):
    "peak and L2 |B_par|/|phi| of (case, field, theta) eigenfunctions; NaN padding counts as 0"
    a = np.abs(np.nan_to_num(ef))
    peak = a[:, 2].max(-1) / a[:, 0].max(-1)
    l2 = np.sqrt(trapezoid(a[:, 2] ** 2, theta, axis=-1) / trapezoid(a[:, 0] ** 2, theta, axis=-1))
    return peak, l2


def phi_overlap(th_s, phi_s, th_g, phi_g):
    "|<phi_GFTM, phi_GS2>| on GS2's theta grid; GFTM interpolated, zero outside its plot range"
    out = []
    for ps, pg in zip(np.nan_to_num(phi_s), phi_g):
        pg = np.interp(th_s, th_g, pg.real, left=0, right=0) + 1j * np.interp(th_s, th_g, pg.imag, left=0, right=0)
        ip = lambda a, b: abs(trapezoid(np.conj(a) * b, th_s))
        # ponytail: max over conjugation, so a sign convention for omega between codes cannot fake a mismatch
        out.append(max(ip(pg, ps), ip(np.conj(pg), ps)) / np.sqrt(ip(pg, pg) * ip(ps, ps)))
    return np.array(out)


def table(db, w, ky, beta, g_s, ef_s, th_s, label, keep, g_g, ef_g, th_g):
    "one row per kept case; all arrays are flattened over cases, ef as (case, field, theta)"
    rs, rs2 = field_ratios(th_s, ef_s)
    rg, rg2 = field_ratios(th_g, ef_g)
    return pd.DataFrame(dict(db=db, width=w, ky=ky, beta=beta, g_gs2=g_s, g_gftm=g_g, kbm=label == "KBM",
                             r_gs2=rs, r_gftm=rg, r2_gs2=rs2, r2_gftm=rg2,
                             overlap=phi_overlap(th_s, ef_s[:, 0], th_g, ef_g[:, 0])))[keep]


def mag(da):
    return np.asarray(da.pint.dequantify().values)


rows = []
for db in hypercubes:
    base = data_root / "GS2" / "Runs" / "LATIN_HYPERCUBE" / db / "kbm_8d"
    scan, last = mc.load_gs2_cube(base, "pyro_cube")         # final-time eigenfunctions
    label = np.asarray(mc.classify(mc.indicators(scan, last)))
    avg = PyroScanGKOutput.from_netcdf(base / "pyro_cube_avg" / "cube.nc").data   # tail-averaged gamma (the hypercube reference)
    names = last.sample_name.values.astype(str)
    assert (avg.sample_name.values.astype(str) == names).all()
    g_s = mag(avg.growth_rate)
    ef_s = mag(last.eigenfunctions.transpose("sample", "field", "theta"))
    assert list(last.field.values) == ["phi", "apar", "bpar"]
    for w, (leaf, _) in widths.items():
        _, d = mc.load_gftm_cube(data_root / "GFTM" / "Runs" / "LATIN_HYPERCUBE" / db / "kbm_8d" / leaf)
        idx = pd.Index(d.sample_name.values.astype(str)).get_indexer(names)
        assert (idx >= 0).all(), f"{db} {leaf}: GS2 samples missing from GFTM"
        d = d.isel(sample=idx)
        assert list(d.field.values) == ["phi", "apar", "bpar"]
        g_g, ef_g = dominant(mag(d.growth_rate.transpose("sample", "mode")),
                             mag(d.eigenfunctions.transpose("sample", "field", "theta", "mode")))
        keep = np.isfinite(g_s) & (g_s > 0)
        rows.append(table(db, w, mag(last.ky), mag(last.beta), g_s, ef_s, mag(last.theta), label, keep,
                          g_g, ef_g, mag(d.theta)))
    print(db, "GS2 unstable", int(keep.sum()), "of", len(g_s), "| GS2 labels", pd.Series(label[keep]).value_counts().to_dict())


def load_scan(code, project, case, leaf):
    d = data_root / code / "Runs" / project / case / leaf
    s = PyroScan(pyroscan_json=d / "pyroscan.json", load_base_pyro=True)
    s.load_gk_output(netcdf_file=d / "pyroscan.nc")
    return s


for case in beta_scans:
    s = mc.attach_legacy_funcs(load_scan("GS2", "scan_beta", case, "parameter_scan_GS2"))
    gs2 = s.gk_output.data
    assert list(gs2.growth_rate.dims) == ["ky", "beta"]
    label = np.asarray(mc.classify(mc.indicators(s, mc.as_samples(s, gs2)))).reshape(gs2.growth_rate.shape).ravel()
    g_s = mag(gs2.growth_rate).ravel()
    tol = mag(gs2.growth_rate_tolerance).ravel()
    ef_s = mag(gs2.eigenfunctions.transpose("ky", "beta", "field", "theta"))
    ef_s = ef_s.reshape(-1, *ef_s.shape[2:])
    ky, beta = (a.ravel() for a in np.meshgrid(mag(gs2.ky), mag(gs2.beta), indexing="ij"))
    for w, (_, leaf) in widths.items():
        d = load_scan("GFTM", "scan_beta_f05", case, leaf).gk_output.data
        d = d.sel(ky=mag(gs2.ky), method="nearest")             # GFTM grid may hold extra ky
        assert np.allclose(mag(d.ky), mag(gs2.ky), atol=1e-3) and d.beta.size == gs2.beta.size
        f = mag(d.beta) / mag(gs2.beta)
        assert np.allclose(f, f[0], rtol=1e-3), "GFTM beta is not GS2 beta x constant"
        g_g, ef_g = dominant(mag(d.growth_rate.transpose("ky", "beta", "mode")).reshape(len(g_s), -1),
                             mag(d.eigenfunctions.transpose("ky", "beta", "field", "theta", "mode")).reshape(len(g_s), 3, d.theta.size, -1))
        keep = np.isfinite(g_s) & (g_s > 0) & (tol < gs2_tol)
        rows.append(table(case, w, ky, beta, g_s, ef_s, mag(gs2.theta), label, keep, g_g, ef_g, mag(d.theta)))
    print(case, "GS2 unstable & converged", int(keep.sum()), "of", len(g_s), "| GS2 labels", pd.Series(label[keep]).value_counts().to_dict())

D = pd.concat(rows, ignore_index=True)
D["R"], D["R2"] = D.r_gftm / D.r_gs2, D.r2_gftm / D.r2_gs2
with np.errstate(divide="ignore"):
    D["e"] = np.where(D.g_gftm > 0, np.log10(D.g_gftm / D.g_gs2), np.nan)   # GFTM finds no unstable mode: e undefined (a MISS)
D["dg"] = (D.g_gftm - D.g_gs2).abs()
D["same"] = (D.e.abs() < np.log10(same_gamma)) & (D.overlap > same_overlap)
print(D.groupby(["width", "db"], sort=False).agg(n=("e", "size"), kbm=("kbm", "sum"), miss=("e", lambda x: x.isna().sum()),
                                                 same=("same", "sum")).to_string())

## Step 1–2: is a large GFTM B∥ a marker of a bad growth rate?
Within each database: top quartile of r_GFTM (and of R) against the rest. Medians of e and |Δγ|, and Spearman ρ(e, log r).

In [ ]:
def large_vs_rest(df, col):
    big = df[col] >= df[col].quantile(top)
    rho = stats.spearmanr(np.log10(df[col]), df.e, nan_policy="omit")[0]
    return pd.Series({"n": len(df), "e_top": df.e[big].median(), "e_rest": df.e[~big].median(),
                      "dg_top": df.dg[big].median(), "dg_rest": df.dg[~big].median(), "rho": rho})


def selection_table(cols):
    out = []
    for pop, sel in {"unstable": D.e.notna(), "KBM": D.e.notna() & D.kbm}.items():
        for (w, db), df in D[sel].groupby(["width", "db"], sort=False):
            for col in cols:
                out.append(large_vs_rest(df, col).rename((pop, w, db, col)))
    return pd.DataFrame(out)


pd.set_option("display.width", 200)
sel_raw = selection_table(["r_gftm", "r2_gftm", "R", "R2"])
print(sel_raw.round(3).to_string())

Figure 1: e against r_GFTM (low-level scatter, all GS2-unstable cases; filled = GS2-KBM). Coloured by ky, and by β.

In [ ]:
def scatter_grid(x, colour, cnorm, clabel, xlabel, fname):
    fig, axes = plt.subplots(len(widths), len(databases), figsize=(4.2 * len(databases), 3.8 * len(widths)), sharey=True,
                             squeeze=False)
    for i, w in enumerate(widths):
        for j, db in enumerate(databases):
            ax, df = axes[i, j], D[(D.width == w) & (D.db == db) & D.e.notna()]
            for kbm, fc in ((False, "none"), (True, None)):
                p = df[df.kbm == kbm]
                sc = ax.scatter(x(p), p.e, c=colour(p), norm=cnorm, cmap="viridis", s=18, marker="o",
                                **({"facecolors": "none"} if fc == "none" else {}))
                if fc == "none":   # hollow markers: edge takes the colour
                    sc.set_edgecolors(plt.cm.viridis(cnorm(colour(p))))
            ax.axhline(0, color="k", lw=0.8, ls=":")
            ax.set_xscale("log")
            ax.tick_params(labelsize=fs_tick)
            if i == 0:
                ax.set_title(db, fontsize=fs_title)
            if i == len(widths) - 1:
                ax.set_xlabel(xlabel, fontsize=fs_label)
        axes[i, 0].set_ylabel(rf"WIDTH {w}" + "\n" + r"$\log_{10}(\gamma_{\rm GFTM}/\gamma_{\rm GS2})$", fontsize=fs_label)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=cnorm, cmap="viridis"), ax=axes, shrink=0.8)
    cb.set_label(clabel, fontsize=fs_label)
    cb.ax.tick_params(labelsize=fs_tick)
    fig.savefig(plot_dir / fname, dpi=150, bbox_inches="tight")
    plt.show()


ky_norm = LogNorm(D.ky.min(), D.ky.max())
rlab = r"$(|B_\parallel|/|\phi|)_{\rm GFTM}$"
scatter_grid(lambda p: p.r_gftm, lambda p: p.ky, ky_norm, r"$k_y\rho$", rlab, "fig1_e_vs_rgftm_ky.png")
scatter_grid(lambda p: p.r_gftm, lambda p: p.beta, Normalize(0, D.beta.max()), r"$\beta$", rlab, "fig1b_e_vs_rgftm_beta.png")

## Step 3: the ky dependence of R on same-mode cases
Fit log10 R = a_db + p·log10 ky per database, and pooled with one intercept per database. 95% CI from the slope's standard
error (t distribution). Then the residual scatter for the fixed exponents p = −1 (the 38e9.9 reading) and −2 (betasig's 1/ky²).

In [ ]:
def fit(df, col="R"):
    "slope p, 95% CI half-width, and residual rms of log10(col) vs log10(ky) with one intercept per database"
    x, y = np.log10(df.ky.values), np.log10(df[col].values)
    G = pd.get_dummies(df.db).values.astype(float)
    A = np.column_stack([x, G])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    res = y - A @ coef
    dof = len(y) - A.shape[1]
    cov = np.linalg.inv(A.T @ A) * (res @ res) / dof
    return coef[0], stats.t.ppf(0.975, dof) * np.sqrt(cov[0, 0]), np.sqrt(res @ res / dof)


def rms_fixed(df, p, col="R"):
    "residual rms of log10(col * ky^-p) about each database's mean: how well a fixed power collapses R"
    z = np.log10(df[col]) - p * np.log10(df.ky)
    return np.sqrt(((z - z.groupby(df.db).transform("mean")) ** 2).mean())


fits = []
for w in widths:
    S = D[(D.width == w) & D.same]
    for col in ("R", "R2"):
        for db, df in [*S.groupby("db", sort=False), ("pooled", S)]:
            if df.ky.nunique() < 3:
                continue
            p, ci, rms = fit(df, col)
            fits.append(dict(width=w, field=col, db=db, n=len(df), p=p, ci=ci, rms_fit=rms, rms_p0=rms_fixed(df, 0, col),
                             rms_pm1=rms_fixed(df, -1, col), rms_pm2=rms_fixed(df, -2, col),
                             medlogRky=np.median(np.log10(df[col] * df.ky))))
fits = pd.DataFrame(fits)
print(fits.round(3).to_string())

Figure 2: R against ky on same-mode cases, with the per-database fitted slope (solid) and a 1/ky line (dashed) through the median.

In [ ]:
fig, axes = plt.subplots(len(widths), len(databases), figsize=(4.2 * len(databases), 3.8 * len(widths)), sharey=True, squeeze=False)
for i, w in enumerate(widths):
    for j, db in enumerate(databases):
        ax, df = axes[i, j], D[(D.width == w) & (D.db == db) & D.same]
        ax.scatter(df.ky, df.R, s=18, c=np.where(df.kbm, "C0", "none"), edgecolors="C0")
        f = fits[(fits.width == w) & (fits.db == db) & (fits.field == "R")]
        if len(f) and len(df):
            k = np.geomspace(df.ky.min(), df.ky.max())
            a = np.median(np.log10(df.R) - f.p.iloc[0] * np.log10(df.ky))
            ax.plot(k, 10 ** a * k ** f.p.iloc[0], "k-", marker="none", label=rf"$k_y^{{{f.p.iloc[0]:.2f}}}$")
            a1 = np.median(np.log10(df.R * df.ky))
            ax.plot(k, 10 ** a1 / k, "k--", marker="none", label=r"$k_y^{-1}$")
            ax.legend(fontsize=fs_legend, loc="lower left")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.tick_params(labelsize=fs_tick)
        if i == 0:
            ax.set_title(db, fontsize=fs_title)
        if i == len(widths) - 1:
            ax.set_xlabel(r"$k_y\rho$", fontsize=fs_label)
    axes[i, 0].set_ylabel(rf"WIDTH {w}" + "\n" + r"$R$ = GFTM / GS2", fontsize=fs_label)
fig.savefig(plot_dir / "fig2_R_vs_ky_samemode.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 3b: does the ky correction remove the large-B∥ / bad-γ association?
Corrected GFTM ratio r_GFTM·ky (the 1/ky hypothesis) and r_GFTM·ky^(−p_pooled) (fitted). Repeat the step-2 selection.

In [ ]:
p_pooled = {w: float(fits[(fits.width == w) & (fits.db == "pooled") & (fits.field == "R")].p.iloc[0]) for w in widths}
print("pooled same-mode exponent p:", {w: round(p, 3) for w, p in p_pooled.items()})
D["r_gftm_ky"] = D.r_gftm * D.ky
D["R_ky"] = D.R * D.ky
D["r_gftm_kyp"] = D.r_gftm * D.ky ** -D.width.map(p_pooled)
D["R_kyp"] = D.R * D.ky ** -D.width.map(p_pooled)
sel_corr = selection_table(["r_gftm_ky", "R_ky", "r_gftm_kyp", "R_kyp"])
print(sel_corr.round(3).to_string())

# collapse check on the same-mode population: spread of log10 R before and after
for w in widths:
    S = D[(D.width == w) & D.same]
    print(f"WIDTH {w}: same-mode n={len(S)} | log10 R median {np.log10(S.R).median():.2f}, IQR {np.log10(S.R).quantile(.75) - np.log10(S.R).quantile(.25):.2f}"
          f" | log10 R*ky median {np.log10(S.R_ky).median():.2f}, IQR {np.log10(S.R_ky).quantile(.75) - np.log10(S.R_ky).quantile(.25):.2f}")

Figure 3: e against the ky-corrected GFTM ratio r_GFTM·ky (all GS2-unstable cases, filled = GS2-KBM), coloured by ky.

In [ ]:
scatter_grid(lambda p: p.r_gftm * p.ky, lambda p: p.ky, ky_norm, r"$k_y\rho$",
             r"$k_y\,(|B_\parallel|/|\phi|)_{\rm GFTM}$", "fig3_e_vs_rgftm_times_ky.png")

## Pooled summary (all databases, per WIDTH)

In [ ]:
summary = []
for w in widths:
    for pop, sel in {"unstable": D.e.notna(), "KBM": D.e.notna() & D.kbm}.items():
        df = D[(D.width == w) & sel]
        for col in ("r_gftm", "R", "r_gftm_ky", "R_ky"):
            big = df[col] >= df.groupby("db")[col].transform(lambda c: c.quantile(top))   # top quartile within each database
            summary.append(dict(width=w, pop=pop, ratio=col, n=len(df), n_top=int(big.sum()),
                                e_top=df.e[big].median(), e_rest=df.e[~big].median(),
                                frac_over2_top=(df.e[big] > np.log10(2)).mean(), frac_over2_rest=(df.e[~big] > np.log10(2)).mean()))
print(pd.DataFrame(summary).round(3).to_string())